In [ ]:
import pandas as pd
import requests
from io import StringIO
from datetime import date, timedelta, datetime

base_url = "https://data.cityofchicago.org/resource/6dvr-xwnh.csv"
start_date = date(2026, 6, 1)
end_date = date(2026, 9, 1)

rows_per_hour = 130
all_chunks = []

current = start_date
while current < end_date:
    for hour in range(24):
        hour_start = datetime.combine(current, datetime.min.time()) + timedelta(hours=hour)
        hour_end = hour_start + timedelta(hours=1)

        params = {
            "$where": f"trip_start_timestamp between '{hour_start.strftime('%Y-%m-%dT%H:%M:%S')}' and '{hour_end.strftime('%Y-%m-%dT%H:%M:%S')}'",
            "$select": "trip_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,"
                       "pickup_community_area,dropoff_community_area,fare,tip,additional_charges,"
                       "trip_total,shared_trip_authorized,trips_pooled",
            "$limit": rows_per_hour,
            "$order": "trip_start_timestamp"
        }
        r = requests.get(base_url, params=params)
        if r.status_code == 200 and r.text.strip():
            chunk = pd.read_csv(StringIO(r.text))
            if len(chunk) > 0:
                all_chunks.append(chunk)
        else:
            print(f"Failed: {current} hour {hour} — status {r.status_code}")

    print(f"{current}: done ({sum(len(c) for c in all_chunks)} total rows so far)")
    current += timedelta(days=1)

df = pd.concat(all_chunks, ignore_index=True)
print(df.shape)

2026-06-01: done (3120 total rows so far)
2026-06-02: done (6240 total rows so far)
2026-06-03: done (9360 total rows so far)
2026-06-04: done (12480 total rows so far)
2026-06-05: done (15600 total rows so far)
2026-06-06: done (18720 total rows so far)
2026-06-07: done (21840 total rows so far)
2026-06-08: done (24960 total rows so far)
2026-06-09: done (28080 total rows so far)
2026-06-10: done (31200 total rows so far)
2026-06-11: done (34320 total rows so far)
2026-06-12: done (37440 total rows so far)
2026-06-13: done (40560 total rows so far)
2026-06-14: done (43680 total rows so far)
2026-06-15: done (46800 total rows so far)
2026-06-16: done (49920 total rows so far)
2026-06-17: done (53040 total rows so far)
2026-06-18: done (56160 total rows so far)
2026-06-19: done (59280 total rows so far)
2026-06-20: done (62400 total rows so far)
2026-06-21: done (65520 total rows so far)
2026-06-22: done (68640 total rows so far)
2026-06-23: done (71760 total rows so far)
2026-06-24: do

In [ ]:
# ============================================
# STEP 1: Basic inspection
# ============================================
print(df.shape)
df.info()
df.isnull().sum()

(286434, 19)
<class 'pandas.core.frame.DataFrame'>
Index: 286434 entries, 0 to 286909
Data columns (total 19 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   trip_id                 286434 non-null  object        
 1   trip_start_timestamp    286434 non-null  datetime64[ns]
 2   trip_end_timestamp      286434 non-null  datetime64[ns]
 3   trip_seconds            286434 non-null  int64         
 4   trip_miles              286434 non-null  float64       
 5   pickup_community_area   286434 non-null  int64         
 6   dropoff_community_area  286434 non-null  int64         
 7   fare                    286434 non-null  float64       
 8   tip                     286434 non-null  float64       
 9   additional_charges      286434 non-null  float64       
 10  trip_total              286434 non-null  float64       
 11  shared_trip_authorized  286434 non-null  bool          
 12  trips_pooled          

,0
trip_id,0
trip_start_timestamp,0
trip_end_timestamp,0
trip_seconds,0
trip_miles,0
pickup_community_area,0
dropoff_community_area,0
fare,0
tip,0
additional_charges,0


In [ ]:
# ============================================
# STEP 2: Convert timestamps
# ============================================
df['trip_start_timestamp'] = pd.to_datetime(df['trip_start_timestamp'])
df['trip_end_timestamp'] = pd.to_datetime(df['trip_end_timestamp'])

In [ ]:
# ============================================
# STEP 3: Remove bad/invalid rows
# ============================================
df = df[df['trip_seconds'] > 0]
df = df[df['trip_miles'] > 0]
df = df[df['fare'] >= 0]
df = df.drop_duplicates(subset='trip_id')

print(df.shape)  # check how many rows survived

In [ ]:
# ============================================
# STEP 4: Handle nulls in community area
# (common — Chicago suppresses these for privacy on short/rare trips)
# ============================================
df['pickup_community_area'] = df['pickup_community_area'].fillna(-1).astype(int)
df['dropoff_community_area'] = df['dropoff_community_area'].fillna(-1).astype(int)

In [ ]:
# ============================================
# STEP 5: Derived columns for analysis
# ============================================
df['hour'] = df['trip_start_timestamp'].dt.hour
df['day_of_week'] = df['trip_start_timestamp'].dt.day_name()
df['date'] = df['trip_start_timestamp'].dt.date
df['is_shared'] = df['shared_trip_authorized'].fillna(False).astype(bool)
df['trip_hours'] = df['trip_seconds'] / 3600
df['avg_speed_mph'] = df['trip_miles'] / df['trip_hours'].replace(0, pd.NA)

In [ ]:
# ============================================
# STEP 6: Verify the date/day spread before moving on
# ============================================
print(df['date'].min(), df['date'].max())
print(df['day_of_week'].value_counts())

2026-06-01 2026-08-31
day_of_week
Monday       43590
Friday       40513
Thursday     40513
Sunday       40508
Wednesday    40465
Tuesday      40439
Saturday     40406
Name: count, dtype: int64


In [ ]:
# ============================================
# STEP 7: Export cleaned CSV
# ============================================
df.to_csv("chicago_tnp_clean.csv", index=False)

In [ ]:
from google.colab import files

files.download("chicago_tnp_clean.csv")